# Week 3 Deep Analysis — Baseline Setup

**Owner:** Member 3 — Jaykisan Saharan  
**Day:** Monday — Baseline and setup

Purpose: initialize the analysis environment, resolve repository-relative paths, and load/inspect the existing order-level KPI exports. These saved exports are interim artifacts; do not treat their values as final until they are reconciled with the validated SQL outputs.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the repository root whether Jupyter starts in the repo root or notebooks/.
CWD = Path.cwd().resolve()
if (CWD / 'data' / 'interim').exists():
    REPO_ROOT = CWD
elif (CWD.parent / 'data' / 'interim').exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        'Could not locate data/interim. Start Jupyter from the repository root or notebooks/.'
    )

INTERIM_DIR = REPO_ROOT / 'data' / 'interim'
print(f'Repository root: {REPO_ROOT}')
print(f'Interim data directory: {INTERIM_DIR}')

In [ ]:
# Load the current saved order-level KPI exports using relative paths.
orders_path = INTERIM_DIR / 'kpi_orders.csv'
orders_all_path = INTERIM_DIR / 'kpi_orders_all.csv'

for path in (orders_path, orders_all_path):
    if not path.exists():
        raise FileNotFoundError(f'Required export not found: {path}')

orders = pd.read_csv(orders_path, low_memory=False)
orders_all = pd.read_csv(orders_all_path, low_memory=False)

print('kpi_orders.csv:', orders.shape)
print('kpi_orders_all.csv:', orders_all.shape)

## Initial structural inspection
Check the columns, data types, and sample rows before calculating new metrics. Keep this first pass descriptive and avoid silently changing source exports.

In [ ]:
display(orders.head())
display(orders.dtypes.rename('dtype').to_frame())
display(orders_all.head())
display(orders_all.dtypes.rename('dtype').to_frame())

In [ ]:
# Basic integrity checks for the order-level exports.
def inspect_export(df: pd.DataFrame, name: str) -> pd.DataFrame:
    summary = pd.DataFrame({
        'check': ['rows', 'columns', 'duplicate_order_ids', 'missing_order_id'],
        'value': [
            len(df),
            len(df.columns),
            int(df['order_id'].duplicated().sum()) if 'order_id' in df.columns else None,
            int(df['order_id'].isna().sum()) if 'order_id' in df.columns else None,
        ],
    })
    print(f'\n{name}')
    display(summary)
    return df.isna().sum().sort_values(ascending=False).rename('missing_values').to_frame()

missing_orders = inspect_export(orders, 'kpi_orders.csv')
missing_orders_all = inspect_export(orders_all, 'kpi_orders_all.csv')
display(missing_orders.head(15).rename_axis('column'))
display(missing_orders_all.head(15).rename_axis('column'))

In [ ]:
# Inspect order statuses and date coverage without assuming every order was delivered.
if 'order_status' in orders_all.columns:
    display(orders_all['order_status'].value_counts(dropna=False).rename_axis('order_status').to_frame('orders'))

if 'order_purchase_timestamp' in orders_all.columns:
    purchase_ts = pd.to_datetime(orders_all['order_purchase_timestamp'], errors='coerce')
    print('Unparseable purchase timestamps:', int(purchase_ts.isna().sum()))
    print('Earliest valid purchase timestamp:', purchase_ts.min())
    print('Latest valid purchase timestamp:', purchase_ts.max())

In [ ]:
# Compare exports at a high level. Differences are evidence to investigate, not an automatic error.
comparison = pd.DataFrame({
    'export': ['kpi_orders.csv', 'kpi_orders_all.csv'],
    'rows': [len(orders), len(orders_all)],
    'unique_order_ids': [orders['order_id'].nunique() if 'order_id' in orders else None, orders_all['order_id'].nunique() if 'order_id' in orders_all else None],
})
display(comparison)

if 'order_id' in orders.columns and 'order_id' in orders_all.columns:
    ids_orders = set(orders['order_id'].dropna().astype(str))
    ids_all = set(orders_all['order_id'].dropna().astype(str))
    print('IDs only in kpi_orders.csv:', len(ids_orders - ids_all))
    print('IDs only in kpi_orders_all.csv:', len(ids_all - ids_orders))

print('Inspection complete. Do not publish final KPIs from these exports until they are reconciled with validated SQL outputs.')

# Week 3 Deep Analysis — Wednesday: Time Trends

Member 3: Jaykisan Saharan

This notebook analyzes monthly delivery-time trends using the repository-relative aggregate export.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data' / 'interim' / 'kpi_orders.csv').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data' / 'interim' / 'kpi_orders.csv'
CHART_PATH = ROOT / 'charts'
CHART_PATH.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'], errors='coerce')
df.shape, df.columns.tolist()

## Monthly trend and data limitations
The source has an order purchase timestamp, delivery duration, and on-time flag. It does not contain a carrier handoff timestamp, so processing-versus-transit time cannot be reliably separated.

In [ ]:
delivered = df[(df['order_status'].eq('delivered')) & df['delivery_days'].notna() & df['order_purchase_timestamp'].notna()].copy()
delivered['month'] = delivered['order_purchase_timestamp'].dt.to_period('M')
monthly = delivered.groupby('month').agg(order_count=('order_id','nunique'), avg_delivery_days=('delivery_days','mean'), on_time_rate=('is_on_time','mean')).reset_index()
monthly['month'] = monthly['month'].astype(str)
monthly

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), layout='constrained')
ax.plot(monthly['month'], monthly['avg_delivery_days'], marker='o', label='Average delivery time')
ax.set(title='Monthly Average Delivery Time', xlabel='Order purchase month', ylabel='Average delivery time (days)')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.25)
ax2 = ax.twinx()
ax2.plot(monthly['month'], monthly['on_time_rate'] * 100, marker='s', linestyle='--', label='On-time rate')
ax2.set_ylabel('On-time delivery rate (%)')
lines1, labels1 = ax.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', frameon=False)
plt.show()

## Findings to validate
- Monthly average delivery time and on-time rate vary over the available months.
- This is descriptive, not causal; monthly order volume varies and should be considered.
- Only delivered orders with valid purchase timestamps and delivery duration are included.
- `processing_vs_transit.png` is not produced because no reliable carrier handoff timestamp is present.

# Week 3 Deep Analysis — Thursday (Day 4): Robustness and Chart Verification

**Owner:** Member 3 — Jaykisan Saharan  
**Day:** Thursday — Robustness and recommendations

This section preserves the Monday baseline setup and Wednesday time-trend work above. It reruns the three project charts from the current interim order-level export, checks their inputs, and records reconciliation status. A chart is **not** described as SQL-validated unless a comparable validated SQL result is available and has been explicitly compared.


In [ ]:
# Thursday: reload the interim source and establish explicit checks before refreshing figures.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

THURSDAY_ROOT = Path.cwd().resolve()
if not (THURSDAY_ROOT / "data" / "interim").exists():
    THURSDAY_ROOT = THURSDAY_ROOT.parent
THURSDAY_DATA = THURSDAY_ROOT / "data" / "interim" / "kpi_orders.csv"
THURSDAY_CHARTS = THURSDAY_ROOT / "charts"
THURSDAY_CHARTS.mkdir(parents=True, exist_ok=True)

if not THURSDAY_DATA.exists():
    raise FileNotFoundError(f"Expected interim export not found: {THURSDAY_DATA}")

thursday_df = pd.read_csv(THURSDAY_DATA, low_memory=False)
required = {"order_id", "order_status", "customer_state", "category", "delivery_days",
            "order_purchase_timestamp", "is_on_time"}
missing = sorted(required - set(thursday_df.columns))
if missing:
    raise ValueError(f"Missing columns required by chart refresh: {missing}")

thursday_df["order_purchase_timestamp"] = pd.to_datetime(
    thursday_df["order_purchase_timestamp"], errors="coerce"
)
thursday_df["delivery_days"] = pd.to_numeric(thursday_df["delivery_days"], errors="coerce")
thursday_df["is_on_time"] = pd.to_numeric(thursday_df["is_on_time"], errors="coerce")
print("Source:", THURSDAY_DATA)
print("Rows:", len(thursday_df))
print("Unique order IDs:", thursday_df["order_id"].nunique())
print("Duplicate order IDs:", int(thursday_df["order_id"].duplicated().sum()))
print("Missing purchase timestamps:", int(thursday_df["order_purchase_timestamp"].isna().sum()))
print("Status counts:")
display(thursday_df["order_status"].value_counts(dropna=False).rename_axis("order_status").to_frame("rows"))


In [ ]:
# Refresh chart 1: late-delivery rate by state. Exclude unknown flags from the denominator.
eligible_state = thursday_df[
    thursday_df["order_status"].eq("delivered")
    & thursday_df["is_on_time"].isin([0, 1])
    & thursday_df["customer_state"].notna()
].copy()
eligible_state = eligible_state.drop_duplicates("order_id")
state_summary = eligible_state.groupby("customer_state").agg(
    eligible_orders=("order_id", "nunique"),
    late_orders=("is_on_time", lambda s: int((s == 0).sum()))
)
state_summary["late_rate_pct"] = 100 * state_summary["late_orders"] / state_summary["eligible_orders"]
state_plot = state_summary.sort_values("late_rate_pct", ascending=False).head(15).sort_values("late_rate_pct")

fig, ax = plt.subplots(figsize=(10, 7), layout="constrained")
ax.barh(state_plot.index.astype(str), state_plot["late_rate_pct"])
ax.set_title("Late Delivery Rate by State — Top 15")
ax.set_xlabel("Late delivery rate (%)")
ax.set_ylabel("Customer state")
ax.grid(axis="x", alpha=0.25)
for y, (_, row) in enumerate(state_plot.iterrows()):
    ax.text(row["late_rate_pct"] + 0.3, y, f'{row["late_rate_pct"]:.1f}%  (n={int(row["eligible_orders"])})', va="center", fontsize=8)
fig.savefig(THURSDAY_CHARTS / "late_rate_by_state.png", dpi=180, bbox_inches="tight")
plt.show()
display(state_summary.sort_values("late_rate_pct", ascending=False).head(15))


In [ ]:
# Refresh chart 2: mean delivery delay by product category; annotate sample size.
eligible_category = thursday_df[
    thursday_df["order_status"].eq("delivered")
    & thursday_df["category"].notna()
    & thursday_df["delivery_days"].notna()
].copy()
eligible_category = eligible_category.drop_duplicates("order_id")
category_summary = eligible_category.groupby("category").agg(
    eligible_orders=("order_id", "nunique"),
    mean_delivery_days=("delivery_days", "mean")
)
category_plot = category_summary.sort_values("mean_delivery_days", ascending=False).head(15).sort_values("mean_delivery_days")

fig, ax = plt.subplots(figsize=(11, 8), layout="constrained")
ax.barh(category_plot.index.astype(str), category_plot["mean_delivery_days"])
ax.axvline(0, linewidth=0.8)
ax.set_title("Average Delivery Duration by Product Category — Top 15")
ax.set_xlabel("Mean delivery duration (days)")
ax.set_ylabel("Product category")
ax.grid(axis="x", alpha=0.25)
for y, (_, row) in enumerate(category_plot.iterrows()):
    ax.text(row["mean_delivery_days"] + 0.15, y, f'{row["mean_delivery_days"]:.1f} d  (n={int(row["eligible_orders"])})', va="center", fontsize=8)
fig.savefig(THURSDAY_CHARTS / "delivery_delay_by_segment.png", dpi=180, bbox_inches="tight")
plt.show()
display(category_summary.sort_values("mean_delivery_days", ascending=False).head(15))


In [ ]:
# Refresh chart 3: monthly delivery duration and on-time rate, with monthly sample counts.
eligible_month = thursday_df[
    thursday_df["order_status"].eq("delivered")
    & thursday_df["delivery_days"].notna()
    & thursday_df["order_purchase_timestamp"].notna()
].copy()
eligible_month = eligible_month.drop_duplicates("order_id")
eligible_month["month"] = eligible_month["order_purchase_timestamp"].dt.to_period("M").astype(str)
monthly_summary = eligible_month.groupby("month").agg(
    eligible_orders=("order_id", "nunique"),
    avg_delivery_days=("delivery_days", "mean"),
    on_time_rate=("is_on_time", "mean")
).reset_index()
monthly_summary["on_time_rate_pct"] = monthly_summary["on_time_rate"] * 100

fig, ax = plt.subplots(figsize=(12, 6), layout="constrained")
ax.plot(monthly_summary["month"], monthly_summary["avg_delivery_days"], marker="o", label="Average delivery duration")
ax.set_title("Monthly Delivery Trend — Interim Export")
ax.set_xlabel("Purchase month")
ax.set_ylabel("Average delivery duration (days)")
ax.tick_params(axis="x", rotation=55)
ax.grid(axis="y", alpha=0.25)
ax2 = ax.twinx()
ax2.plot(monthly_summary["month"], monthly_summary["on_time_rate_pct"], marker="s", linestyle="--", label="On-time rate")
ax2.set_ylabel("On-time delivery rate (%)")
l1, lab1 = ax.get_legend_handles_labels()
l2, lab2 = ax2.get_legend_handles_labels()
ax.legend(l1 + l2, lab1 + lab2, loc="best", frameon=False)
fig.savefig(THURSDAY_CHARTS / "monthly_delivery_trend.png", dpi=180, bbox_inches="tight")
plt.show()
display(monthly_summary)


## Thursday verification findings and limitations

- The three figures above are regenerated from `data/interim/kpi_orders.csv`; this is still an interim source.
- State late-rate denominators include only delivered orders with a known binary `is_on_time` flag.
- Category comparisons use delivered orders with a non-missing numeric `delivery_days` value; sample counts are shown to make small groups visible.
- Monthly figures use delivered orders with valid purchase timestamps and delivery duration. The on-time rate is calculated from available `is_on_time` values; check the denominator before publication.
- The source exports have previously shown different row counts. Do not interpret that difference as a confirmed error without reconciling order IDs and inclusion rules.
- `processing_vs_transit.png` remains ungenerated because no validated carrier handoff timestamp is available in the analysis export.
- **SQL reconciliation status:** pending until these chart populations, numerators, denominators, and metric definitions are compared with the team's validated SQL output. Do not label these charts SQL-verified before that comparison.


# Week 3 Deep Analysis — Friday (Day 5): Final Review and Handover

**Owner:** Member 3 — Jaykisan Saharan  
**Day:** Friday — Final review and handover

This section completes the Python analysis handover. It reloads the repository's interim order-level export, runs final integrity checks, regenerates the three agreed charts, and records their source definitions and limitations. Earlier Monday, Wednesday, and Thursday sections are retained above.

**Important:** These outputs are reproducible from the interim CSV. They are not described as SQL-validated unless the matching validated SQL aggregates have been compared separately.


In [ ]:
# Friday final review: reload source, verify required fields, and inspect order-level integrity.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

FINAL_ROOT = Path.cwd().resolve()
if not (FINAL_ROOT / "data" / "interim").exists():
    FINAL_ROOT = FINAL_ROOT.parent
FINAL_DATA = FINAL_ROOT / "data" / "interim" / "kpi_orders.csv"
FINAL_CHARTS = FINAL_ROOT / "charts"
FINAL_CHARTS.mkdir(parents=True, exist_ok=True)

if not FINAL_DATA.exists():
    raise FileNotFoundError(f"Cannot find interim KPI export: {FINAL_DATA}")

final_df = pd.read_csv(FINAL_DATA, low_memory=False)
required_columns = {
    "order_id", "order_status", "customer_state", "category",
    "delivery_days", "order_purchase_timestamp", "is_on_time"
}
missing_columns = sorted(required_columns - set(final_df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

final_df["order_purchase_timestamp"] = pd.to_datetime(
    final_df["order_purchase_timestamp"], errors="coerce"
)
final_df["delivery_days"] = pd.to_numeric(final_df["delivery_days"], errors="coerce")
final_df["is_on_time"] = pd.to_numeric(final_df["is_on_time"], errors="coerce")

integrity_summary = pd.DataFrame({
    "check": [
        "source rows", "unique order IDs", "duplicate order IDs",
        "missing order IDs", "invalid purchase timestamps",
        "missing delivery duration", "unknown/non-binary on-time flags"
    ],
    "value": [
        len(final_df),
        final_df["order_id"].nunique(),
        int(final_df["order_id"].duplicated().sum()),
        int(final_df["order_id"].isna().sum()),
        int(final_df["order_purchase_timestamp"].isna().sum()),
        int(final_df["delivery_days"].isna().sum()),
        int((~final_df["is_on_time"].isin([0, 1])).sum())
    ]
})
display(integrity_summary)
display(final_df["order_status"].value_counts(dropna=False).rename_axis("order_status").to_frame("rows"))

# Use one row per order for final descriptive chart aggregates. Duplicate IDs are reported above.
final_orders = final_df.dropna(subset=["order_id"]).drop_duplicates("order_id").copy()
print("Final chart source:", FINAL_DATA)
print("Rows in source:", len(final_df))
print("Distinct orders used after de-duplication:", final_orders["order_id"].nunique())


In [ ]:
# Final chart 1: state late-delivery rate with explicit numerator and denominator.
state_population = final_orders[
    final_orders["order_status"].eq("delivered")
    & final_orders["is_on_time"].isin([0, 1])
    & final_orders["customer_state"].notna()
].copy()
final_state = state_population.groupby("customer_state").agg(
    eligible_orders=("order_id", "nunique"),
    late_orders=("is_on_time", lambda s: int((s == 0).sum()))
)
final_state["late_rate_pct"] = 100 * final_state["late_orders"] / final_state["eligible_orders"]
state_top15 = final_state.nlargest(15, "late_rate_pct").sort_values("late_rate_pct")

fig, ax = plt.subplots(figsize=(10, 7), layout="constrained")
ax.barh(state_top15.index.astype(str), state_top15["late_rate_pct"])
ax.set(title="Late Delivery Rate by State — Top 15",
       xlabel="Late delivery rate (%)", ylabel="Customer state")
ax.grid(axis="x", alpha=0.25)
for y, (_, row) in enumerate(state_top15.iterrows()):
    ax.text(row["late_rate_pct"] + 0.25, y,
            f'{row["late_rate_pct"]:.1f}% (n={int(row["eligible_orders"]):,})',
            va="center", fontsize=8)
fig.savefig(FINAL_CHARTS / "late_rate_by_state.png", dpi=180, bbox_inches="tight")
plt.show()
display(final_state.sort_values("late_rate_pct", ascending=False).head(15))


In [ ]:
# Final chart 2: average delivery duration by category with sample counts.
category_population = final_orders[
    final_orders["order_status"].eq("delivered")
    & final_orders["category"].notna()
    & final_orders["delivery_days"].notna()
].copy()
final_category = category_population.groupby("category").agg(
    eligible_orders=("order_id", "nunique"),
    mean_delivery_days=("delivery_days", "mean")
)
category_top15 = final_category.nlargest(15, "mean_delivery_days").sort_values("mean_delivery_days")

fig, ax = plt.subplots(figsize=(11, 8), layout="constrained")
ax.barh(category_top15.index.astype(str), category_top15["mean_delivery_days"])
ax.axvline(0, linewidth=0.8)
ax.set(title="Average Delivery Duration by Product Category — Top 15",
       xlabel="Mean delivery duration (days)", ylabel="Product category")
ax.grid(axis="x", alpha=0.25)
for y, (_, row) in enumerate(category_top15.iterrows()):
    value = row["mean_delivery_days"]
    offset = 0.15 if value >= 0 else -0.15
    ax.text(value + offset, y, f'{value:.1f} d (n={int(row["eligible_orders"]):,})',
            va="center", ha="left" if offset > 0 else "right", fontsize=8)
fig.savefig(FINAL_CHARTS / "delivery_delay_by_segment.png", dpi=180, bbox_inches="tight")
plt.show()
display(final_category.sort_values("mean_delivery_days", ascending=False).head(15))


In [ ]:
# Final chart 3: monthly delivery duration and on-time rate with separate valid denominators.
monthly_population = final_orders[
    final_orders["order_status"].eq("delivered")
    & final_orders["delivery_days"].notna()
    & final_orders["order_purchase_timestamp"].notna()
].copy()
monthly_population["month"] = monthly_population["order_purchase_timestamp"].dt.to_period("M").astype(str)
final_monthly = monthly_population.groupby("month").agg(
    eligible_delivery_orders=("order_id", "nunique"),
    avg_delivery_days=("delivery_days", "mean"),
    on_time_orders=("is_on_time", lambda s: int((s == 1).sum())),
    on_time_denominator=("is_on_time", lambda s: int(s.isin([0, 1]).sum()))
).reset_index()
final_monthly["on_time_rate_pct"] = (
    final_monthly["on_time_orders"]
    .div(final_monthly["on_time_denominator"].replace(0, float("nan"))) * 100
)

fig, ax = plt.subplots(figsize=(12, 6), layout="constrained")
ax.plot(final_monthly["month"], final_monthly["avg_delivery_days"],
        marker="o", label="Average delivery duration")
ax.set(title="Monthly Delivery Trend — Final Interim Export",
       xlabel="Purchase month", ylabel="Average delivery duration (days)")
ax.tick_params(axis="x", rotation=55)
ax.grid(axis="y", alpha=0.25)
ax2 = ax.twinx()
ax2.plot(final_monthly["month"], final_monthly["on_time_rate_pct"],
         marker="s", linestyle="--", label="On-time rate")
ax2.set_ylabel("On-time delivery rate (%)")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="best", frameon=False)
fig.savefig(FINAL_CHARTS / "monthly_delivery_trend.png", dpi=180, bbox_inches="tight")
plt.show()
display(final_monthly)


## Friday final review — outputs and limitations

- The three charts are regenerated from `data/interim/kpi_orders.csv` by the cells above.
- Each chart uses distinct `order_id` values after duplicate IDs are reported and removed for the descriptive calculations.
- State late-rate denominator: delivered orders with a binary `is_on_time` value (0 or 1); numerator: those with value 0.
- Category metric: mean `delivery_days` for delivered orders with a non-missing numeric duration. This is delivery duration, not a separately validated delay-versus-estimate metric.
- Monthly delivery duration uses delivered orders with valid purchase timestamps and non-missing `delivery_days`. Monthly on-time rate uses only rows with binary `is_on_time` values; its denominator is shown separately.
- The available source is an interim KPI export. Differences between `kpi_orders.csv` and `kpi_orders_all.csv`, if present, must be reconciled by order ID and inclusion rule.
- **SQL reconciliation remains pending** until matching validated SQL aggregates are compared for populations, counts, numerators, denominators, and metric definitions.
- `processing_vs_transit.png` is not produced because a reliable carrier handoff timestamp is not established in the available source.
- Findings are descriptive and do not establish causes. Small monthly or segment samples can produce unstable averages/rates.
- Final handover files: this notebook, the three PNG charts, and `reports/chart_notes.md`.
